In [1]:
import pandas as pd
import json
import os,time
from openai import OpenAI #estamos la clase concreta OpenAI del módulo openai
from dotenv import load_dotenv #importamos una función concreta del módulo
load_dotenv("template.env")

# Acceder a la clave de API de OpenAI
api_key = os.getenv("OPENAI_API_KEY")

# Asegurarte de que la clave de API se haya cargado correctamente
if api_key is None:
    raise ValueError("La clave de API no está configurada en el archivo .env")
    
client = OpenAI() #creando un objeto de la clase

f_t_job = client.fine_tuning.jobs.retrieve("ftjob-rHxNZYkt5qaQ3fnxKCRnwIOO")

fine_tuned_model_id = f_t_job.fine_tuned_model

In [2]:
#PROMPTS

#AGE PROMPT
categorize_system_prompt_paraphrase ='''
La edad de adquisición (AoA) de una palabra se refiere a la edad en la que se aprendió una palabra por primera vez. 
En concreto, cuándo una persona habría entendido por primera vez esa palabra si alguien la hubiera utilizado delante de ella, incluso cuando aún no la hubiera dicho, leído o escrito. 
Calcule la edad media de adquisición (AoA) de la palabra {palabra} para un hablante nativo de español.

El formato de salida debe ser un objeto JSON: {AoA: número //AoA de la palabra expresado en años, puede incluir decimales, Word: palabra //string}
'''

In [ ]:
#FUNCTION DECLARATION
def get_line_file(file_name,line,extract_func):
	with open(file_name, 'r') as f:
		for line_number, theline in enumerate(f):
			if line_number == line:
				res = theline
				break
	res = json.loads(res)
	return extract_func(res)

def extract_data(new_line):
	res = new_line["response"]["body"]["choices"][0]["message"]["content"]
	res = json.loads(res)
	return res

def create_file_from_tasks(tasks,file_name):
	with open(file_name, 'w') as file:
		for obj in tasks:
			file.write(json.dumps(obj) + '\n')


def create_batch(file_name):
	batch_file = client.files.create(
		file = open(file_name, "rb"),
		purpose = "batch"
	)
	batch_job = client.batches.create(
		input_file_id = batch_file.id,
		endpoint = "/v1/chat/completions",
		completion_window = "24h"
	)
	return batch_job

def generate_task(index,prompt,word,model):
	task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "temperature": 0,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [
                {
                    "role": "user",
                    "content": word_into_prompt(prompt,word)
                }
            ],
        }
    }
	return task

#DIVIDE TASK
def divide_task(tasks_array,file_array,task_index,num_tasks):
	res_task_array = []
	res_file_array = []
	task_array_to_div = []
	prov_file_name = ""
	for i in range(0,len(tasks_array)):
		if(i == task_index):
			task_array_to_div = tasks_array[i]
			prov_file_name = file_array[i]
		else:
			res_task_array.append(tasks_array[i])
			res_file_array.append(file_array[i])
	index = 1+int(len(task_array_to_div)/num_tasks)
	for i in range(0,index):
		tasks = []
		for j in range(0,num_tasks):
			if(i*num_tasks+j < len(task_array_to_div)):
				tasks.append(task_array_to_div[i*num_tasks+j])
		res_task_array.append(tasks)
		res_file_array.append(prov_file_name.replace(".json","_"+"0"*(1+int(index/10)-len(str(i)))+str(i)+".json"))
	return res_task_array,res_file_array

def create_task_from_json(json_object,index,prompt,model):
	word = json_object["Word"]
	task = generate_task(
		index,prompt,word,model,
	)
	return task

def create_task_array_from_dataframe(df,prompt,model):
	tasks = []
	for index, row in df.iterrows():
		task = create_task_from_json(row,index,prompt,model)
		tasks.append(task)
	return tasks

def word_into_prompt(prompt,word):
	return prompt.replace("{palabra}",word)

In [4]:
dataset_folder = os.getenv("DATASET_FOLDER")
dataset_path = str(dataset_folder) + "FinalResults-AoA-no-FT-prompt-JSON.xlsx"

df = pd.read_excel(dataset_path)
df = df.sample(30)
df.head()

,Word,AoA,Source
53383,esprit,5.0,fundeu_dic
109531,sanedrín,12.5,Dictionary_on_web
1404,acelajado,12.5,RAE_dic
66891,honorable,10.5,Dictionary_on_web
85992,ñandubay,10.5,Dictionary_on_web


In [5]:
tasks_array = [create_task_array_from_dataframe(df,categorize_system_prompt_paraphrase,fine_tuned_model_id)]
#tasks_array = [create_task_array_from_dataframe(df,categorize_system_prompt_paraphrase,"gpt-4o-mini")]
file_array = ["middle_files/batch_job_mmlu_age.jsonl"]

tasks_array,file_array = divide_task(tasks_array,file_array,0,10000)

#GENERATE TASK FILES
for i in range(0,len(tasks_array)):
	create_file_from_tasks(tasks_array[i],file_array[i])
	print(file_array[i])

middle_files/batch_job_mmlu_age_0.jsonl


In [9]:
#GENERATE BATCH
batch_jobs = []
for i in range(0,len(tasks_array)):
	ba_jo = create_batch(file_array[i])
	batch_jobs.append(ba_jo)

In [12]:
#COMPLETION_CHECK
status_num = 0
while (status_num != len(batch_jobs)):
	for i in range(0,60):
		time.sleep(1)
	status_num = 0
	for i in range(0,len(batch_jobs)):
		batch = batch_jobs[i]
		batch = client.batches.retrieve(batch.id)
		#print(batch)
		result_file_id = batch.output_file_id
		status = batch.status
		if (status == "completed"):
			status_num += 1
		#print(status)
print("success")


success


In [ ]:
#CANCEL BATCH
batch_id = client.batches.retrieve(batch_jobs[0].id).id
client.batches.cancel(batch_id)

Batch(id='batch_67ecec91a99c8190ad4619c2b5dfbcc2', completion_window='24h', created_at=1743580305, endpoint='/v1/chat/completions', input_file_id='file-9fU4VM8LHp8niucHaLptMi', object='batch', status='cancelling', cancelled_at=None, cancelling_at=1743599056, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1743666705, failed_at=None, finalizing_at=None, in_progress_at=1743580307, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=10))

In [13]:
#OUTPUT FILES GENERATOR
print("")
for i in range(0,len(batch_jobs)):
	batch = batch_jobs[i]
	batch = client.batches.retrieve(batch.id)
	result_file_id = batch.output_file_id

	result = client.files.content(result_file_id).content

	result_file_name = file_array[i].replace(".json","_result.json")

	with open(result_file_name, 'wb') as file:
		file.write(result)

In [14]:
def outfile(file_array):
	rows = []
	errors = []
	acum = 0

	for i in range(0,len(file_array)):
		f_a = file_array[i].replace(".json","_result.json")
		with open(f_a, 'r') as f:
			for line in f:
				row = df.iloc[acum]
				try:
					dt_a = extract_data(json.loads(line.strip()))
					newAoA = dt_a["AoA"]
				except:
					newAoA = "NaN"
					print(f"file_num {i}\nline_num {acum}\nline {line}")
					errors.append({"Line_Num":acum})
				rows.append({
      		  		"Word":row['Word'],
					"FT_AoA":newAoA,
					"not_FT_AoA":row["AoA"],
					"Source":row['Source']
				})
				acum += 1
	return pd.DataFrame(rows), pd.DataFrame(errors)

file_name = "output_files/Results_AoA_f_t_2000.xlsx"
clean_dtset,errors_dtset = outfile(file_array)

with pd.ExcelWriter(file_name) as writer:
	clean_dtset.to_excel(writer, sheet_name='Results',index=False)
	errors_dtset.to_excel(writer, sheet_name='Errors',index=False)

file_num 0
line_num 0
line {"id": "batch_req_67f28b21e9ac819085680d24e95240ef", "custom_id": "task-53383", "response": {"status_code": 200, "request_id": "816131e4db509f78891c528e042f5b57", "body": {"id": "chatcmpl-BJKhwRu9tkPsnWcJHx9X8gWZXHfZe", "object": "chat.completion", "created": 1743947532, "model": "ft:gpt-4o-mini-2024-07-18:ging-upm::B9GdDHzb", "choices": [{"index": 0, "message": {"role": "assistant", "content": "                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [118]:
def extract_input(new_line):
	return (new_line["body"]["messages"])

test_file = "middle_files/batch_job_mmlu_age_0.jsonl"
test_line = get_line_file(test_file,0,extract_input)

test_messages = []
test_messages.append(test_line[0])
test_messages.append(test_line[1])
response = client.chat.completions.create(
	model=fine_tuned_model_id, messages=test_messages, temperature=0
)

print(test_messages)
print(response.choices[0].message.content)

[{'role': 'system', 'content': '\nLa edad de adquisición (AoA) de una palabra se refiere a la edad en la que se aprendió una palabra por primera vez. \nEn concreto, cuándo una persona habría entendido por primera vez esa palabra si alguien la hubiera utilizado delante de ella, incluso cuando aún no la hubiera dicho, leído o escrito. \nCalcule la edad media de adquisición (AoA) de la palabra {palabra} para un hablante nativo de español.\n\nEl formato de salida debe ser un objeto JSON: {AoA: número //AoA de la palabra expresado en años, puede incluir decimales, Word: palabra //string}\n'}, {'role': 'user', 'content': 'radioescucha'}]
9.8
